# Hybrid Movie Recommendation System

A movie recommendation engine built on the MovieLens dataset, combining two complementary
approaches into a single hybrid model:

- **Collaborative Filtering** (Matrix Factorization / Funk-SVD) — learns from user rating
  behavior to find patterns across similar users and movies.
- **Content-Based Filtering** (TF-IDF + Cosine Similarity) — recommends movies based on
  genre similarity to a user's rating history.

The two models are combined with a tunable weighted average, with the blend weight
selected on a held-out validation set and final performance reported on an untouched
test set.

**Dataset:** MovieLens (`movies.csv`: 9,742 movies · `ratings.csv`: 100,836 ratings from 610 users)

--------------------------------------------------------------------------------------
## 1. Data Loading & Exploration
Loading the MovieLens dataset (movies.csv, ratings.csv) and reviewing basic statistics.

In [23]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

print(movies.shape, ratings.shape)
print(movies.head())
print(ratings.head())

(9742, 3) (100836, 4)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


## 2. Train / Validation / Test Split
A per-user split ensures every user has ratings in all three sets, avoiding cold-start bias during evaluation.

In [24]:
def train_test_split_per_user(ratings, test_frac=0.2, min_ratings_to_split=5, seed=42):
    rng = np.random.RandomState(seed)
    train_idx, test_idx = [], []

    for user_id, group in ratings.groupby('userId'):
        idx = group.index.to_numpy().copy()
        
        if len(idx) < min_ratings_to_split:
            train_idx.extend(idx)
            continue
        
        rng.shuffle(idx)
        n_test = max(1, int(len(idx) * test_frac))
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])

    train = ratings.loc[train_idx].reset_index(drop=True)
    test = ratings.loc[test_idx].reset_index(drop=True)
    return train, test

In [25]:
train_full, test = train_test_split_per_user(ratings, test_frac=0.2)
train, val = train_test_split_per_user(train_full, test_frac=0.15)

print(f"train={len(train)}  val={len(val)}  test={len(test)}")

train=69042  val=11854  test=19940


## 3. Collaborative Filtering — Matrix Factorization (Funk-SVD)
Learns latent factors for each user and movie via stochastic gradient descent (SGD), trained only on observed ratings.

In [26]:
class CollaborativeModel:
    
    def __init__(self, n_factors=30, lr=0.01, reg=0.05, n_epochs=15, seed=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.seed = seed

    def fit(self, train_ratings):
        rng = np.random.RandomState(self.seed)

        self.user_ids = train_ratings['userId'].unique()
        self.item_ids = train_ratings['movieId'].unique()
        self.user_idx = {u: i for i, u in enumerate(self.user_ids)}
        self.item_idx = {m: i for i, m in enumerate(self.item_ids)}
     
        n_users, n_items = len(self.user_ids), len(self.item_ids)
        self.global_mean = train_ratings['rating'].mean()

        self.P = rng.normal(0, 0.1, (n_users, self.n_factors))
        self.Q = rng.normal(0, 0.1, (n_items, self.n_factors))
        self.bu = np.zeros(n_users)
        self.bi = np.zeros(n_items)

        u_arr = train_ratings['userId'].map(self.user_idx).to_numpy()
        i_arr = train_ratings['movieId'].map(self.item_idx).to_numpy()
        r_arr = train_ratings['rating'].to_numpy()

        for epoch in range(self.n_epochs):
            order = rng.permutation(len(r_arr))
            sq_err_sum = 0.0

            for k in order:
                    u, i, r = u_arr[k], i_arr[k], r_arr[k]
                    pred = self.global_mean + self.bu[u] + self.bi[i] + self.P[u] @ self.Q[i]
                    err = r - pred
                    sq_err_sum += err ** 2

                    self.bu[u] += self.lr * (err - self.reg * self.bu[u])
                    self.bi[i] += self.lr * (err - self.reg * self.bi[i])
                    p_u = self.P[u].copy()
                    self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                    self.Q[i] += self.lr * (err * p_u - self.reg * self.Q[i])

            rmse = np.sqrt(sq_err_sum / len(r_arr))
            print(f"epoch {epoch + 1}/{self.n_epochs} - train RMSE: {rmse:.4f}")

        return self


    def predict(self, user_id, movie_id):
        if user_id not in self.user_idx or movie_id not in self.item_idx:
            return self.global_mean
        u, i = self.user_idx[user_id], self.item_idx[movie_id]
        pred = self.global_mean + self.bu[u] + self.bi[i] + self.P[u] @ self.Q[i]
        return float(np.clip(pred, 0.5, 5.0))

### Training & Evaluating the Collaborative Model

In [27]:
cf = CollaborativeModel(n_factors=30, lr=0.01, reg=0.05, n_epochs=15)
cf.fit(train)

epoch 1/15 - train RMSE: 0.9478
epoch 2/15 - train RMSE: 0.8911
epoch 3/15 - train RMSE: 0.8693
epoch 4/15 - train RMSE: 0.8549
epoch 5/15 - train RMSE: 0.8437
epoch 6/15 - train RMSE: 0.8342
epoch 7/15 - train RMSE: 0.8256
epoch 8/15 - train RMSE: 0.8176
epoch 9/15 - train RMSE: 0.8091
epoch 10/15 - train RMSE: 0.8009
epoch 11/15 - train RMSE: 0.7922
epoch 12/15 - train RMSE: 0.7830
epoch 13/15 - train RMSE: 0.7731
epoch 14/15 - train RMSE: 0.7626
epoch 15/15 - train RMSE: 0.7515


In [29]:
sample_user = train['userId'].iloc[0]
sample_movie = train['movieId'].iloc[0]

prediction = cf.predict(sample_user, sample_movie)
print(f"User: {sample_user}, Movie: {sample_movie}")
print(f"Model Prediction: {prediction:.2f}")

User: 1, Movie: 3729
Model Prediction: 4.20


In [30]:
def evaluate(model, test_ratings):
    preds = np.array([model.predict(u, m) for u, m in zip(test_ratings['userId'], test_ratings['movieId'])])
    actual = test_ratings['rating'].to_numpy()
    rmse = np.sqrt(np.mean((preds - actual) ** 2))
    return rmse
    
test_rmse = evaluate(cf, test)
print(f"RMSE Model on test data: {test_rmse:.4f}")

RMSE Model on test data: 0.8896


## 4. Content-Based Filtering — Genre Similarity
Represents each movie as a TF-IDF vector over its genres, builds a per-user taste profile, and scores movies via cosine similarity.

In [31]:
class ContentBasedModel:
    def __init__(self):
        pass

    def fit(self, movies, train_ratings):
        self.movies = movies.reset_index(drop=True)
        self.movie_id_to_row = {mid: row for row, mid in enumerate(self.movies['movieId'])}
        genre_corpus = self.movies['genres'].str.replace('|', ' ', regex=False)
        self.vectorizer = TfidfVectorizer(token_pattern=r'[^\s]+')
        self.genre_matrix = self.vectorizer.fit_transform(genre_corpus)

        self.user_profiles = {}
        user_means = train_ratings.groupby('userId')['rating'].mean()
        for user_id, group in train_ratings.groupby('userId'):
            rows = [self.movie_id_to_row[m] for m in group['movieId'] if m in self.movie_id_to_row]
            if not rows:
                continue
            weights = (group['rating'] - user_means[user_id]).to_numpy()
            weights = weights[:len(rows)]
            vecs = self.genre_matrix[rows]
            if np.all(weights == 0):
                weights = np.ones_like(weights)
            profile = np.asarray(vecs.T @ weights).flatten()
            norm = np.linalg.norm(profile)
            if norm > 0:
                profile = profile / norm
            self.user_profiles[user_id] = (profile, user_means[user_id])

        self.global_mean = train_ratings['rating'].mean()
        return self

    def predict(self, user_id, movie_id):
        if user_id not in self.user_profiles or movie_id not in self.movie_id_to_row:
            return self.global_mean
        profile, user_mean = self.user_profiles[user_id]
        row = self.movie_id_to_row[movie_id]
        movie_vec = self.genre_matrix[row].toarray().flatten()
        sim = cosine_similarity(profile.reshape(1, -1), movie_vec.reshape(1, -1))[0, 0]
        pred = user_mean + sim * 2.0
        return float(np.clip(pred, 0.5, 5.0))

### Training & Evaluating the Content-Based Model

In [32]:
cb = ContentBasedModel()
cb.fit(movies, train)
print(len(cb.user_profiles))

610


In [33]:
cb_rmse = evaluate(cb, test)
print(f"RMSE Mpdel on content-based on tested data: {cb_rmse:.4f}")

RMSE Mpdel on content-based on tested data: 1.0145


## 5. Hybrid Model — Weighted Combination
Combines both models as `alpha * CF + (1 - alpha) * CB`. Alpha is tuned on the validation set to avoid leaking test data into model selection.

In [35]:
class HybridModel:
    def __init__(self, cf_model, cb_model, alpha=0.7):
        self.cf_model = cf_model
        self.cb_model = cb_model
        self.alpha = alpha

    def predict(self, user_id, movie_id):
        cf_pred = self.cf_model.predict(user_id, movie_id)
        cb_pred = self.cb_model.predict(user_id, movie_id)
        return self.alpha * cf_pred + (1 - self.alpha) * cb_pred

    def tune_alpha(self, val_ratings, alphas=np.arange(0.0, 1.01, 0.1)):
        best_alpha, best_rmse = self.alpha, np.inf
        for a in alphas:
            self.alpha = a
            rmse = evaluate(self, val_ratings)
            print(f"alpha={a:.1f} -> RMSE={rmse:.4f}")
            if rmse < best_rmse:
                best_rmse, best_alpha = rmse, a
        self.alpha = best_alpha
        print(f"Best alpha = {best_alpha:.1f} با RMSE={best_rmse:.4f}")
        return best_alpha

### Tuning Alpha & Final Test Evaluation

In [36]:
hybrid = HybridModel(cf, cb, alpha=0.7)
hybrid.tune_alpha(val)

alpha=0.0 -> RMSE=1.0116
alpha=0.1 -> RMSE=0.9774
alpha=0.2 -> RMSE=0.9470
alpha=0.3 -> RMSE=0.9207
alpha=0.4 -> RMSE=0.8989
alpha=0.5 -> RMSE=0.8819
alpha=0.6 -> RMSE=0.8701
alpha=0.7 -> RMSE=0.8636
alpha=0.8 -> RMSE=0.8626
alpha=0.9 -> RMSE=0.8671
alpha=1.0 -> RMSE=0.8769
Best alpha = 0.8 با RMSE=0.8626


np.float64(0.8)

In [37]:
final_rmse = evaluate(hybrid, test)
print(f"RMSE final model hybrid (alpha={hybrid.alpha:.1f}) on test data: {final_rmse:.4f}")

RMSE final model hybrid (alpha=0.8) on test data: 0.8741


## 6. Top-N Recommendation Function
Generates a ranked list of unseen movies for a given user based on predicted ratings.

In [38]:
def recommend_for_user(model, user_id, movies, train_ratings, n=10):
    seen = set(train_ratings.loc[train_ratings['userId'] == user_id, 'movieId'])
    candidates = movies.loc[~movies['movieId'].isin(seen), 'movieId']
    scores = [(mid, model.predict(user_id, mid)) for mid in candidates]
    scores.sort(key=lambda x: x[1], reverse=True)
    top = scores[:n]
    result = movies.set_index('movieId').loc[[mid for mid, _ in top], ['title', 'genres']].copy()
    result['predicted_rating'] = [s for _, s in top]
    return result.reset_index()

In [39]:
recommend_for_user(hybrid, user_id=1, movies=movies, train_ratings=train_full, n=10)

,movieId,title,genres,predicted_rating
0,475,In the Name of the Father (1993),Drama,4.998239
1,1276,Cool Hand Luke (1967),Drama,4.990613
2,318,"Shawshank Redemption, The (1994)",Crime|Drama,4.976012
3,306,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama,4.954791
4,1223,"Grand Day Out with Wallace and Gromit, A (1989)",Adventure|Animation|Children|Comedy|Sci-Fi,4.924028
5,1225,Amadeus (1984),Drama,4.919863
6,5690,Grave of the Fireflies (Hotaru no haka) (1988),Animation|Drama|War,4.913150
7,1221,"Godfather: Part II, The (1974)",Crime|Drama,4.910143
8,1104,"Streetcar Named Desire, A (1951)",Drama,4.908805
9,899,Singin' in the Rain (1952),Comedy|Musical|Romance,4.908159


## 7. Ranking Quality — Precision@10
RMSE alone doesn't measure recommendation quality. Precision@10 checks how many of the top-10 recommended movies the user actually rated highly in the test set.

In [40]:
def precision_at_k(model, test_ratings, train_ratings, movies, k=10, rating_threshold=4.0):
    liked_in_test = test_ratings[test_ratings['rating'] >= rating_threshold]
    users_to_eval = liked_in_test['userId'].unique()
    precisions = []
    for user_id in users_to_eval:
        recs = recommend_for_user(model, user_id, movies, train_ratings, n=k)
        recommended_ids = set(recs['movieId'])

        user_liked = set(liked_in_test.loc[liked_in_test['userId'] == user_id, 'movieId'])
        n_hits = len(recommended_ids & user_liked)

        precisions.append(n_hits / k)

    return np.mean(precisions)

    precisions = []
    for user_id in users_to_eval:
        recs = recommend_for_user(model, user_id, movies, train_ratings, n=k)
        recommended_ids = set(recs['movieId'])

        user_liked = set(liked_in_test.loc[liked_in_test['userId'] == user_id, 'movieId'])
        n_hits = len(recommended_ids & user_liked)

        precisions.append(n_hits / k)

    return np.mean(precisions)

In [41]:
sample_users_test = test[test['userId'].isin(test['userId'].unique()[:50])]

precision = precision_at_k(hybrid, sample_users_test, train_full, movies, k=10)
print(f"Precision@10 (on 50 user sample): {precision:.4f}")

Precision@10 (on 50 user sample): 0.0327


In [45]:
cf_precision = precision_at_k(cf, sample_users_test, train_full, movies, k=10)
print(f"Precision@10 (ONLY Collaborative): {cf_precision:.4f}")

Precision@10 (ONLY Collaborative): 0.0265


In [46]:
cb_precision = precision_at_k(cb, sample_users_test, train_full, movies, k=10)
print(f"Precision@10 (ONLY Content-Based): {cb_precision:.4f}")

Precision@10 (ONLY Content-Based): 0.0184


## Summary

This project builds a hybrid movie recommendation system combining two complementary approaches:

- **Collaborative Filtering**: Matrix factorization (Funk-SVD) trained via SGD, learning latent user/movie embeddings and bias terms from the observed rating patterns.
- **Content-Based Filtering**: TF-IDF vectorization of movie genres combined with cosine similarity against a per-user taste profile.

Data was split per-user into train/validation/test sets to prevent user leakage. The hybrid weight (alpha) was tuned exclusively on the validation set, and final performance was reported on a held-out test set never used during model selection.

**Results on the test set:**

| Model | RMSE | Precision@10 |
|---|---|---|
| Content-Based | 1.0145 | 0.0184 |
| Collaborative Filtering | 0.8896 | 0.0265 |
| **Hybrid (alpha = 0.8)** | **0.8741** | **0.0327** |

The hybrid model outperformed both individual approaches on rating-prediction accuracy (RMSE) *and* recommendation ranking quality (Precision@10) — demonstrating that behavioral signals (collaborative) and content signals (genre similarity) capture complementary information neither method captures alone.

**Tech stack:** Python, NumPy, pandas, scikit-learn (TF-IDF, cosine similarity)